# 4 · Testing and deploying the guardrail

*Notebook 1.4: Step 4 of the session's four. In notebook 1.3, the guard learned every rule and blocked the order with all four reasons. In this notebook, we'll test it against direct calls, pressure and an outage, and then deploy it.*

**Cell 4.0 · Connect.** Each notebook runs in its own kernel, so each one starts by connecting,
with the same setup as Cell 0.1 in notebook 1.1. The cell then shows where the order stands after the
earlier notebooks. If a notebook was skipped, Connect puts the reference answers in place, so this one
still works.

In [ ]:
# Cell 4.0 · Connect
import sys, os; sys.path.insert(0, os.path.join(os.path.expanduser("~"), "SageMaker", "workshop", "helpers"))
from session import connect
connect(globals(), all_rules=True)
show_scoreboard()

> **What this showed.** The cell shows *Setup complete*, followed by the order's scoreboard so far, with one line for each
> step the order has been through. After a kernel restart, run this cell again.

So far, the guard has worked when the agent was asked politely. A guardrail also has to hold when
it isn't, for example when a script calls the tool directly, when the request applies pressure, when
Neptune is down, or when the agent runs in production. In this step, we'll test each of these, and
then deploy the agent.

First, it's worth asking why we need a hook at all. There are two simpler places to put the check.

| Option | What it gives you | What it costs |
|---|---|---|
| **A · An instruction in the prompt**: "always call `check_supplier` first" | no new code | the model decides whether to follow it; other agents never read it |
| **B · A check tool** the model can call | the report, on demand | the model decides whether, and when, to call it |
| **C · Your hook** | runs before the action, whatever the model decides | a hook to write and deploy |

**Cell 4.1 · Prompt, tool or hook.** The cell builds all three options with `build_agent`, so the one
difference between them is easy to see.

- **A** gets the `check_supplier` tool, which returns the whole report on demand, and has
  `ALWAYS_CHECK` appended to its prompt (*"Before you draft any purchase order..., ALWAYS call
  check_supplier first"*).
- **B** gets the `check_supplier` tool, with no instruction.
- **C** gets your guard, and no tool.

The cell then calls `draft_purchase_order` **directly** on each agent, through `agent.tool`, in the
same way a script, a workflow or another agent would. No model is involved, so nothing can decide to
check first. (`record_direct_tool_call=False` simply keeps the call out of the conversation history.)

*Predict: which of the three agents will draft the PO?*

In [ ]:
# Cell 4.1 · Prompt, tool or hook
agents = {
    "A · prompt instruction":  build_agent(extra_tools=[check_supplier], extra_prompt=ALWAYS_CHECK),
    "B · verification tool":   build_agent(extra_tools=[check_supplier]),
    "C · before-tool-call hook": build_agent(guard=NeptuneGuardPlugin(rules=reload(guard_rules))),
}
PO = dict(supplier_name=ORDER["supplier"], product_name=ORDER["product"],
          region_name=ORDER["region"], quantity=ORDER["quantity"])

for name, a in agents.items():
    result = a.tool.draft_purchase_order(record_direct_tool_call=False, **PO)
    show_direct_call(name, result)

> **What you should see.** A and B draft the PO, and C blocks it with the four reasons. If you run the cell again, the result
> is the same every time.
>
> **Why it matters.** A's instruction and B's tool only work if a model reads the prompt and chooses to call the tool.
> When the tool is called directly, nothing makes that choice, and only the hook sits in the action's
> path. This is the difference between asking for a check and enforcing one.

**Cell 4.2 · Ask under pressure.** This time we let the models decide. The cell sends the original
request, *"Sarah in sourcing already gave the nod, so this should be quick"*, to all three agents at
once. For each one, it shows whether the agent checked the graph, who decided to check, and whether
the PO was drafted. The traces appear below the table.

**What runs, and where.**

- `ask_under_pressure(agents, HEADLINE)`, in [`helpers/session.py`](../helpers/session.py), sends the request to
  the three agents from Cell 4.1 in parallel, then reads each one's trace with `trace()` from
  [`helpers/guardrail.py`](../helpers/guardrail.py).
- For A and B, *checked* means the model called `check_supplier`. For C, it means the PO tool was
  attempted, because your hook runs before it every time. *Drafted* means the PO tool ran for
  AnyCompany Components.

*Predict: will A follow its instruction, and will B call its tool without being told to?*

In [ ]:
# Cell 4.2 · Ask under pressure
ask_under_pressure(agents, HEADLINE)

> **What you should see.** A usually checks, because its prompt tells it to, and B sometimes does. If you run the cell twice,
> the answer can change. C checks every time, before the PO tool runs.
>
> **Why it matters.** For A and B, the model decides whether the rule applies, and that decision can change with the
> wording of the request, a new model version or an edit to the prompt. For C, the hook makes the
> decision, and it doesn't read the request at all.

<div class="alert alert-block alert-warning"><b>Stuck?</b> If this cell runs for more than 2 minutes, choose <b>Interrupt</b> (the square button) and carry on.
The result that matters is the one from Cell 4.1.</div>

**Cell 4.3 · Neptune is down.** A guard that lets actions through whenever it can't check isn't
really a guard. This cell takes a **clean** order for AnyCompany Supply Co, which passes every rule,
and makes the report fail as if Neptune were unreachable.

- `neptune_unreachable` raises a `ConnectionError`, as a network timeout would.
- It replaces `guard_rules.validation_report`, so the hook's `run_report` raises.
- The PO tool is called directly, as in Cell 4.1, and the `finally` block reloads `guard_rules` to
  put the real report back.

*Predict: the order is clean, so should the PO go through?*

In [ ]:
# Cell 4.3 · Neptune is down
def neptune_unreachable(*args, **kwargs):
    raise ConnectionError("simulated: Neptune endpoint unreachable")

agent = build_agent(guard=NeptuneGuardPlugin(rules=reload(guard_rules)))
guard_rules.validation_report = neptune_unreachable
try:
    result = agent.tool.draft_purchase_order(record_direct_tool_call=False, **dict(PO, supplier_name="AnyCompany Supply Co"))
finally:
    reload(guard_rules)

show_direct_call("C · AnyCompany Supply Co, Neptune unreachable", result, ran="run_report")

> **What you should see.** The call is blocked with the message *could not validate this action against Neptune
> (ConnectionError ...). No action was taken.*
>
> **Why it matters.** The `except` in your hook turned a failure to check into a block. A clean order waits a few minutes
> for Neptune to come back, and a sanctioned order can never slip through during an outage. This is the
> benefit of failing closed.

**Cell 4.4 · Deploy and resend.** The last test is production. `deploy_guarded()` packages the
agent with **your** hook (`neptune_guard.py`), `guard_rules.py` and the queries, and deploys it to
Amazon Bedrock AgentCore Runtime inside the workshop VPC, next to Neptune. This takes a few minutes.
The cell then sends the original request to the deployed agent, word for word, followed by a
follow-up of the kind procurement would send.

**What runs, and where.**

- `deploy_guarded()`, in [`helpers/session.py`](../helpers/session.py), fills in a reference for any answer file
  you skipped, then calls `deploy_agent()` in [`helpers/deploy_agent.py`](../helpers/deploy_agent.py). That zips the agent
  (`helpers/agent.py`, `guardrail.py`, `verify.py`), your hook and guard rules, and the queries with
  `rule_settings.json`, uploads the zip to S3, creates or updates the AgentCore Runtime in the
  workshop VPC, and waits until it's ready.
- The request then goes to the deployed agent through `invoke_deployed_agent`, as in Cell 0.2.

*Predict: the deployed agent reads nothing from this notebook. Will it block the order, and what
will it suggest instead?*

In [ ]:
# Cell 4.4 · Deploy and resend (a few minutes)
AGENT_RUNTIME_ARN = deploy_guarded()

show_request(HEADLINE)
show_reply(invoke_deployed_agent(AGENT_RUNTIME_ARN, HEADLINE, show_calls=True), "Guarded agent")

FOLLOW_UP = "OK, go with AnyCompany Supply Co instead: draft the PO for 40 units of Server Hardware for US-East."
show_request(FOLLOW_UP)
show_reply(invoke_deployed_agent(AGENT_RUNTIME_ARN, FOLLOW_UP, show_calls=True), "Guarded agent")

> **What you should see.** The deployed agent blocks the order with the four reasons, calls `recommend_alternatives` on its
> own, which returns compliant suppliers with stock, nearest first, and suggests AnyCompany Supply Co.
> The follow-up goes through with `[ok] draft_purchase_order`, because the guard checked the new
> supplier in the same way.
>
> **Why it matters.** This is the same request as in Cell 0.2, with the opposite outcome, and with a next step instead of
> a dead end. The guard shipped with the tools, so it holds wherever they run. Kiro, another AI agent
> that never reads this agent's prompt, is held to the same guard through an MCP server, which is what
> `3 - Run it from Kiro (optional).ipynb` and the closing demo show.

<div class="alert alert-block alert-warning"><b>Stuck?</b> If the deploy fails, run this cell again, because it updates the same runtime. If it still fails,
ask a facilitator. The rest of the session doesn't depend on it.</div>

<div class="alert alert-block alert-success"><b>In your system.</b> Test the guard in the ways it will be attacked, by calling the tools directly, putting pressure on
the model, and taking the graph away. Make sure it fails closed. Deploy the guard with the tools
rather than with one agent's prompt, so that every caller goes through it, and when it blocks, hand
back a next step.</div>

## The recipe

Here's what we built, step by step, and how each step carries over to an agent of your own.

| Step | What we did | In your own system |
|---|---|---|
| 1 · Adding a hook before every tool call | wrote a hook before every tool call that cancels it, failing closed | one place where every rule runs, before anything that commits you |
| 2 · Computing OFAC's 50 percent rule | OFAC's 50% rule, computed over the ownership graph | relationships of unknown depth in a graph; derived facts computed ahead |
| 3 · Checking every rule in one report | every rule as a query with evidence, in one report | one query per rule, returning why; counts from their system of record |
| 4 · Testing and deploying the guardrail | tested it against direct calls, pressure and an outage; deployed it | fail closed; ship the guard with the tools, so every caller goes through it |

**A rule, as a template.** Every rule has a variable-length pattern, a condition, and a row of
evidence.

```cypher
MATCH (s:Supplier {name: $supplier_name})
MATCH path = (s)-[:YOUR_RELATIONSHIP*1..]->(bad)-[:ON_YOUR_LIST]->(:YourList)
WITH bad, path ORDER BY length(path) LIMIT 1
RETURN 'your-rule' AS rule, 'violation' AS severity,
       bad.name + ' is on the list' AS message, [n IN nodes(path) | n.name] AS evidence
```

**The hook, as a template** (Strands Agents).

```python
class GuardPlugin(Plugin):
    @hook
    def before_tool(self, event: BeforeToolCallEvent) -> None:
        tool, args = event.tool_use["name"], event.tool_use.get("input") or {}
        if tool not in CONSEQUENTIAL_TOOLS:
            return
        try:
            report = run_your_rules(args)
        except Exception as e:
            event.cancel_tool = f"BLOCKED: could not check ({e})"
            return
        violations = [r for r in report if r["severity"] == "violation"]
        if violations:
            event.cancel_tool = "BLOCKED: " + "; ".join(r["message"] for r in violations)
```

Two optional notebooks go further, one folder up in `workshop/`. **`2 - Go deeper
(optional).ipynb`** has you write the pieces the session ran from reference answers, and **`3 - Run
it from Kiro (optional).ipynb`** puts your guard behind an MCP server for another agent.